# `changemyview` Climate-Relevance Audit / `changemyview` 气候相关性审计

This notebook uses simple counts and title inspection only. It does not use clustering, classification, or stance scores.

本 notebook 只使用简单计数和标题检查，不使用聚类、分类或立场分数。

The goal is to determine whether `changemyview` contains enough clearly climate- or energy-related discussions for the main clustering analysis.

目标是判断 `changemyview` 是否包含足够多明确与气候或能源相关的讨论，从而适合进入主要聚类分析。

## 1. Load the eligible `changemyview` records / 1. 读取符合基础条件的记录

Use the same time window and minimum comment length as Notebook 04. Only the required columns are loaded from the cleaned Parquet file.

使用与04相同的时间范围和最少20词条件，并且只读取本次审计需要的列。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

def display_full_titles(frame):
    styled = (
        frame.style
        .set_properties(
            subset=["post_title"],
            **{
                "white-space": "normal",
                "text-align": "left",
                "min-width": "650px",
                "max-width": "1000px",
            },
        )
        .set_properties(
            subset=[column for column in frame.columns if column != "post_title"],
            **{"white-space": "nowrap"},
        )
    )
    display(styled)

BASE_DIR = Path(r"D:\DM1_project")
DATA_PATH = BASE_DIR / "reddit_opinion_climate_change_cleaned_final.parquet"

START_DATE = pd.Timestamp("2023-11-19")
END_EXCLUSIVE = pd.Timestamp("2026-09-01")
MIN_WORDS = 20
RANDOM_STATE = 42

assert DATA_PATH.exists(), f"Input file not found: {DATA_PATH}"

dataset = ds.dataset(DATA_PATH, format="parquet")
filter_expression = (
    (ds.field("subreddit") == "changemyview")
    & (ds.field("created_time") >= START_DATE.to_pydatetime())
    & (ds.field("created_time") < END_EXCLUSIVE.to_pydatetime())
    & (ds.field("comment_word_count") >= MIN_WORDS)
)

columns = [
    "comment_id", "post_id", "post_title", "clean_text",
    "created_time", "comment_word_count",
]
cmv = dataset.to_table(columns=columns, filter=filter_expression).to_pandas()
cmv = cmv.dropna(subset=["comment_id", "post_id", "clean_text"]).copy()
cmv = cmv.drop_duplicates("comment_id", keep="first")
cmv["post_title"] = cmv["post_title"].fillna("").astype(str).str.strip()

print("Eligible comments / 符合基础条件的评论:", f"{len(cmv):,}")
print("Unique posts / 不同帖子:", f"{cmv['post_id'].nunique():,}")
print("Date range / 时间范围:", cmv["created_time"].min(), "to", cmv["created_time"].max())

Eligible comments / 符合基础条件的评论: 146,912
Unique posts / 不同帖子: 1,847
Date range / 时间范围: 2023-11-19 18:30:03 to 2026-08-31 17:05:04


The eligible set is the complete `changemyview` pool before climate-title screening. The original source file is not modified.

这里得到的是执行气候标题筛选之前的完整 `changemyview` 候选语料，原始文件不会被修改。

## 2. Compare the Notebook 04 pattern with a stricter pattern / 2. 比较04规则和修正后的严格规则

The original pattern matches the standalone word `climate`, so phrases such as `political climate` can pass. The stricter version requires a more explicit climate or energy expression.

04的原始规则可以匹配单独的 `climate`，因此 `political climate` 也可能通过。修正版要求标题出现更明确的气候或能源表达。

In [2]:
ORIGINAL_PATTERN = (
    r"(?i)\b(?:climat(?:e|ic|ology|ologist|ologists)|global\s+warming|"
    r"greenhouse\s+(?:gas|gases|effect)|carbon\s+(?:emission|emissions|tax|capture|budget)|"
    r"co2|methane|net\s+zero|decarbon\w*|fossil\s+fuels?|renewable\s+energy|"
    r"clean\s+energy|solar\s+(?:energy|power|panel|panels)|wind\s+(?:energy|power|turbine|turbines)|"
    r"coal|oil\s+(?:industry|companies|company|production|drilling)|natural\s+gas|"
    r"energy\s+(?:transition|policy|crisis|system|systems|market|markets|grid|security)|"
    r"electric\s+vehicles?|evs?|nuclear\s+(?:power|energy)|reactors?|uranium|"
    r"sea\s+level|ice\s+sheets?|glaciers?|heat\s*waves?|droughts?|wildfires?|"
    r"climate\s+(?:policy|action|change|crisis|science|denial|denier|deniers))\b"
)

STRICT_PATTERN = (
    r"(?i)\b(?:climat(?:ic|ology|ologist|ologists)|global\s+warming|"
    r"greenhouse\s+(?:gas|gases|effect)|"
    r"carbon\s+(?:emission|emissions|tax|capture|budget|neutrality)|"
    r"co2|methane|net\s+zero|decarbon\w*|fossil\s+fuels?|renewable\s+energy|"
    r"clean\s+energy|solar\s+(?:energy|power|panel|panels)|"
    r"wind\s+(?:energy|power|turbine|turbines)|coal|"
    r"oil\s+(?:industry|companies|company|production|drilling)|natural\s+gas|"
    r"energy\s+(?:transition|policy|crisis|system|systems|market|markets|grid|security)|"
    r"electric\s+vehicles?|evs?|nuclear\s+(?:power|energy)|reactors?|uranium|"
    r"sea\s+level|ice\s+sheets?|glaciers?|heat\s*waves?|droughts?|wildfires?|"
    r"climate\s+(?:policy|action|change|crisis|science|denial|denier|deniers|"
    r"emergency|target|targets|mitigation|adaptation|impact|impacts|risk|risks|"
    r"justice|activism|activist|activists))\b"
)

AMBIGUOUS_CLIMATE_PATTERN = (
    r"(?i)\b(?:political|social|economic|business|financial|investment|"
    r"workplace|work|academic|cultural|current)\s+climate\b|"
    r"\bclimate\s+(?:control|controlled)\b"
)

cmv["original_match"] = cmv["post_title"].str.contains(
    ORIGINAL_PATTERN, regex=True, na=False
)
cmv["strict_match"] = cmv["post_title"].str.contains(
    STRICT_PATTERN, regex=True, na=False
)
cmv["ambiguous_climate_phrase"] = cmv["post_title"].str.contains(
    AMBIGUOUS_CLIMATE_PATTERN, regex=True, na=False
)

posts = (
    cmv.groupby(["post_id", "post_title"], as_index=False)
    .agg(
        comments=("comment_id", "size"),
        original_match=("original_match", "first"),
        strict_match=("strict_match", "first"),
        ambiguous_climate_phrase=("ambiguous_climate_phrase", "first"),
    )
)

def audit_row(label, comment_mask, post_mask):
    return {
        "criterion / 标准": label,
        "comments / 评论": int(comment_mask.sum()),
        "comment_percent / 评论比例": round(comment_mask.mean() * 100, 2),
        "posts / 帖子": int(post_mask.sum()),
        "post_percent / 帖子比例": round(post_mask.mean() * 100, 2),
    }

audit_summary = pd.DataFrame([
    audit_row(
        "All eligible / 全部候选",
        pd.Series(True, index=cmv.index),
        pd.Series(True, index=posts.index),
    ),
    audit_row(
        "Original Notebook 04 title match / 04原始标题匹配",
        cmv["original_match"], posts["original_match"],
    ),
    audit_row(
        "Strict climate-energy title match / 严格气候能源标题匹配",
        cmv["strict_match"], posts["strict_match"],
    ),
    audit_row(
        "No strict title match / 未通过严格标题匹配",
        ~cmv["strict_match"], ~posts["strict_match"],
    ),
    audit_row(
        "Ambiguous non-environmental climate phrase / 歧义性非环境climate表达",
        cmv["ambiguous_climate_phrase"], posts["ambiguous_climate_phrase"],
    ),
])

display(audit_summary)

,criterion / 标准,comments / 评论,comment_percent / 评论比例,posts / 帖子,post_percent / 帖子比例
0,All eligible / 全部候选,146912,100.00,1847,100.00
1,Original Notebook 04 title match / 04原始标题匹配,11367,7.74,139,7.53
2,Strict climate-energy title match / 严格气候能源标题匹配,9787,6.66,117,6.33
3,No strict title match / 未通过严格标题匹配,137125,93.34,1730,93.67
4,Ambiguous non-environmental climate phrase / 歧义性非环境climate表达,915,0.62,9,0.49


The strict-match percentage is an operational relevance measure, not proof that every unmatched comment is irrelevant. The title examples below provide direct, inspectable evidence.

严格匹配比例是一种操作性相关指标，并不能证明所有未匹配评论都无关。下面的实际标题为判断提供可以直接核查的证据。

## 3. Inspect the largest discussions that fail the strict rule / 3. 检查未通过严格规则的最大讨论

Large discussions have the greatest potential to dominate comment-level clustering, so they are shown first.

大型讨论最可能主导评论层面的聚类，因此优先展示。

In [3]:
largest_unmatched = (
    posts.loc[~posts["strict_match"], ["post_id", "post_title", "comments"]]
    .sort_values(["comments", "post_title"], ascending=[False, True])
    .head(30)
    .reset_index(drop=True)
)

display_full_titles(largest_unmatched)

,post_id,post_title,comments
0,1i0aqbe,CMV: the world would be a better place without Religion,293
1,183nztu,"CMV: Liberals think conservatives will, or ought to, have an ""Are We the Baddies?"" moment.",284
2,1knqoo9,"CMV: Willful ignorance is destroying America, and it’s time we call it what it is instead of pretending it’s just a “difference of opinion.”",270
3,1li24km,"CMV: People who genuinely believe in things like ghosts, bigfoot, zodiacs, etc. should be treated with the same criticism as those who believe the earth is flat",257
4,1n17v95,CMV: Neither burning the Koran nor so called 'Islamophobic Speech should be illegal acts in the UK (or any other civilised country),255
5,1evrkc0,CMV: I don't really understand why people care so much about Israel-Palestine,253
6,1hxynly,CMV: It's not reasonable to vote on and pass laws that pertain to and restrict the freedoms of a group of people who have no right to vote on them.,253
7,1l03rz5,"CMV: The World Would Be Better Off Without Abrahamic Religions (Islam, Christianity, Judaism)",252
8,1m5ukdm,"CMV: Christians, based on their own teachings, should lean left politically.",251
9,1ocyxrh,CMV: If the Christian God is real then he is very upset at Trump and MAGA.,250


If the displayed high-volume titles concern religion, elections, war, culture, or other non-climate issues, they are direct evidence that the source subreddit contains substantial off-topic material for this research question.

如果这些高评论量标题主要涉及宗教、选举、战争、文化或其他非气候问题，它们就是该社区对本研究问题包含大量题外内容的直接证据。

## 4. Inspect false positives caused by ambiguous uses of `climate` / 4. 检查 `climate` 歧义造成的误匹配

These posts pass the original Notebook 04 pattern but contain expressions such as `political climate` or `social climate`.

这些帖子能够通过04的原始规则，但包含 `political climate` 或 `social climate` 等非环境含义。

In [4]:
ambiguous_false_positives = (
    posts.loc[
        posts["original_match"]
        & ~posts["strict_match"]
        & posts["ambiguous_climate_phrase"],
        ["post_id", "post_title", "comments"],
    ]
    .sort_values(["comments", "post_title"], ascending=[False, True])
    .reset_index(drop=True)
)

print(
    "Ambiguous posts passing the original rule / "
    "通过原始规则的歧义帖子:",
    f"{len(ambiguous_false_positives):,}",
)
print(
    "Comments under these posts / 这些帖子下的评论:",
    f"{ambiguous_false_positives['comments'].sum():,}",
)
display_full_titles(ambiguous_false_positives.head(30))

Ambiguous posts passing the original rule / 通过原始规则的歧义帖子: 9
Comments under these posts / 这些帖子下的评论: 915


,post_id,post_title,comments
0,1e3y6nw,"CMV: The Trump assassination attempt was the natural end result of America's current political climate, and things will only get worse from here.",211
1,1oct61a,CMV: Ranked choice voting is an obvious solution to the polarized political climate in the US.,189
2,1loab2i,CMV: The US Debt problem cannot be fixed in today’s political climate,121
3,1mglqbh,"CMV: The current political climate is only the beginning of a monumental shift towards conservative and extremist politics within Western or all of Europe, that will fully reshape society compared to how we know it today.",104
4,1dhb7zo,CMV: The American people are to blame for the current political climate in the US,81
5,1j5daq5,cmv: I think working out and workout culture is propaganda. This has nothing to do with the current political climate. This is just something I think about a lot,70
6,1d8a1tt,"CMV: in today’s political climate, you can’t marry someone without even slightly agreeing with even their most extreme views",65
7,1p0sunt,CMV: Calling for the boycott of AI for ethical/sustainable reasons is exacerbating polarisation in an already tense political climate and ultimately will do more bad than good,46
8,1nnc3vb,CMV: countries shouldn’t be banned from sports/events due to political climate.,28


This section demonstrates why matching the standalone word `climate` is insufficient for a high-precision climate-change corpus.

这一部分说明：如果要构建高精度气候变化语料，仅匹配单独的 `climate` 并不充分。

## 5. Reproducible random title sample / 5. 可复现的随机标题样本

The following sample is selected with a fixed random seed. It avoids showing only hand-picked extreme examples.

下面使用固定随机种子抽样，避免只展示人为挑选的极端例子。

In [5]:
unmatched_posts = posts.loc[
    ~posts["strict_match"],
    ["post_id", "post_title", "comments"],
]

random_unmatched_sample = (
    unmatched_posts.sample(
        n=min(40, len(unmatched_posts)),
        random_state=RANDOM_STATE,
    )
    .sort_values("comments", ascending=False)
    .reset_index(drop=True)
)

display_full_titles(random_unmatched_sample)

,post_id,post_title,comments
0,1d2nft0,"CMV: Elon Musk is just simply a walking, talking, and tweeting example of the Monopoly Experiments",196
1,1p4ny8f,CMV: American conservatism today is really just an ideology of excusing yourself from solving problems.,185
2,1chylvp,CMV: the meat industry should not be subsidized (non-vegan),174
3,1dz4x64,CMV: There is no legitimate or illegitimate mechanism by which Jan 6th 2021 (or the events leading up to or following it) could have ever possibly resulted in Donald Trump remaining president,166
4,1dw7szc,CMV: Imprisoning CEOs of companies that hire illegal immigrants would effectively end most illegal immigration. The fact that any policy like this hasn't been proposed is proof that neither American party wants to actually address the issue.,164
5,1niqfzj,CMV: Republicans are Stealing From Us,151
6,1ocij9b,"CMV: Legal, brown-skinned immigrants in the USA are at a substantially high risk of having their lives torn apart in the near future.",148
7,1qfi1xb,CMV: Eternal life is something NO ONE should want,122
8,1gui7sk,CMV: You can’t be an intelligent person and vote GOP,122
9,1g5lmsd,CMV: I think that Earth is the only place in the universe where life exists.,115


The fixed random sample provides a transparent manual check of the excluded pool. Readers can judge the titles directly instead of relying only on aggregate percentages.

固定随机样本为被排除语料提供透明的人工核查方式，读者可以直接判断标题，而不必只依赖总体比例。

## 6. Concise conclusion / 6. 简要结论

Automatically report the main evidence while preserving the methodological limitation.

自动汇总主要证据，同时保留方法限制说明。

In [6]:
strict_comment_rate = cmv["strict_match"].mean()
strict_post_rate = posts["strict_match"].mean()
unmatched_comment_rate = 1 - strict_comment_rate
unmatched_post_rate = 1 - strict_post_rate

conclusion = pd.DataFrame({
    "metric / 指标": [
        "Eligible comments / 候选评论",
        "Eligible posts / 候选帖子",
        "Strict-match comments / 严格匹配评论",
        "Strict-match posts / 严格匹配帖子",
        "Comments under unmatched titles / 未匹配标题下的评论",
        "Posts with unmatched titles / 未匹配标题的帖子",
    ],
    "value / 数值": [
        f"{len(cmv):,}",
        f"{len(posts):,}",
        f"{cmv['strict_match'].sum():,} ({strict_comment_rate:.2%})",
        f"{posts['strict_match'].sum():,} ({strict_post_rate:.2%})",
        f"{(~cmv['strict_match']).sum():,} ({unmatched_comment_rate:.2%})",
        f"{(~posts['strict_match']).sum():,} ({unmatched_post_rate:.2%})",
    ],
})
display(conclusion)

print(
    "Conclusion / 结论:\n"
    "Only a small share of changemyview records meets the strict title-based "
    "climate-energy criterion. The largest unmatched discussions and the fixed "
    "random sample contain many directly inspectable non-climate topics. This "
    "supports either excluding changemyview from the main climate clustering "
    "analysis or applying a clearly documented strict relevance filter.\n\n"
    "只有较小比例的changemyview记录满足严格的气候／能源标题标准。"
    "最大未匹配讨论和固定随机样本中可以直接观察到许多非气候主题。"
    "因此，主聚类分析可以考虑排除changemyview，或者对其应用明确记录的严格相关性筛选。"
)

print(
    "Limitation / 限制:\n"
    "A title-based rule cannot prove that every unmatched comment is irrelevant "
    "or that every matched comment is relevant. The displayed titles provide "
    "audit evidence, while a manually labelled random sample would be required "
    "for a formal relevance-accuracy estimate.\n\n"
    "标题规则不能证明每条未匹配评论都无关，也不能证明每条匹配评论都相关。"
    "当前标题展示提供审计证据；如果需要正式估计相关性准确率，还应对随机样本进行人工标注。"
)

,metric / 指标,value / 数值
0,Eligible comments / 候选评论,"146,912"
1,Eligible posts / 候选帖子,"1,847"
2,Strict-match comments / 严格匹配评论,"9,787 (6.66%)"
3,Strict-match posts / 严格匹配帖子,117 (6.33%)
4,Comments under unmatched titles / 未匹配标题下的评论,"137,125 (93.34%)"
5,Posts with unmatched titles / 未匹配标题的帖子,"1,730 (93.67%)"


Conclusion / 结论:
Only a small share of changemyview records meets the strict title-based climate-energy criterion. The largest unmatched discussions and the fixed random sample contain many directly inspectable non-climate topics. This supports either excluding changemyview from the main climate clustering analysis or applying a clearly documented strict relevance filter.

只有较小比例的changemyview记录满足严格的气候／能源标题标准。最大未匹配讨论和固定随机样本中可以直接观察到许多非气候主题。因此，主聚类分析可以考虑排除changemyview，或者对其应用明确记录的严格相关性筛选。
Limitation / 限制:
A title-based rule cannot prove that every unmatched comment is irrelevant or that every matched comment is relevant. The displayed titles provide audit evidence, while a manually labelled random sample would be required for a formal relevance-accuracy estimate.

标题规则不能证明每条未匹配评论都无关，也不能证明每条匹配评论都相关。当前标题展示提供审计证据；如果需要正式估计相关性准确率，还应对随机样本进行人工标注。
